# Harness Engineering 动手实验

![Harness Engineering 课程封面](https://zhangyeio.github.io/assets/images/deepseek-harness-lab/cover.png)

> **模型决定智能的上限，Harness 决定智能能不能稳定地进入真实世界。**

> **范围说明：** 本 Notebook 是帮助初学者理解 Harness 核心概念的教学实现。`MiniKernel`、`SessionPlugin`、`ToolPlugin` 等均为课程自建示例，并不是 DeepSeek Harness SDK 的正式 API，也不代表生产系统的完整实现。

本实验默认使用确定性的 `mock` 模式，不需要 API Key，也不会发送网络请求。只有显式设置 `HARNESS_LAB_MODE=live` 时，才会调用 DeepSeek API。

读这个 Notebook 不需要先会框架。你只需要知道：我们可以用 Python 调用大模型。

学完后，不要求你记住所有类和函数，只要能讲清楚三句话：

1. **Agent 不是一次模型调用**，而是模型根据工具结果不断决定下一步的循环。
2. **Tool Calls 只是调用申请**，真正执行工具的是模型外面的程序。
3. **Harness 不负责生成主要业务结论**，它决定模型看到什么、能做什么、何时继续或停止，以及如何记录和验收。

整份 Notebook 按六级台阶推进，每一级只增加一个新东西：

![从聊天到 Harness 的六级学习台阶](https://zhangyeio.github.io/assets/images/deepseek-harness-lab/01-learning-staircase.svg)

请按顺序逐格运行。后面的代码会用到前面定义的函数和变量。


## 第一站只认识 5 个词

不要一开始背完整术语表。第一部分只需要认识下面五个词，其他工程术语留到真正需要时再讲。

把它们想成**做一顿晚饭**：家人说「今晚想吃番茄炒蛋」，五个词分别站在厨房的不同位置。

| 词 | 生活里可以这样想 |
|---|---|
| **模型** | 会想菜的人：听得懂「今晚吃番茄炒蛋」，但隔着门看不见冰箱。不打开，就只能猜「应该有鸡蛋」 |
| **工具（Tool）** | 打开冰箱、拉开橱柜：人说「去看」，手才真正打开，返回「有番茄，没鸡蛋」 |
| **Context** | 贴在冰箱上的便签：家人想吃什么、已经看到了什么。每轮只能根据便签上这些做决定 |
| **Loop** | 看完一样就写回便签，再看一眼：还要继续翻，还是可以定菜单 |
| **Agent** | 不是随口一答，而是「想菜 + 开冰箱 + 根据里面有什么再决定」这一整次做饭 |

![模型、工具、Context、Loop 与 Agent 的晚饭类比](https://zhangyeio.github.io/assets/images/deepseek-harness-lab/00-five-words.svg)


## 0. 环境准备

本实验分成两种运行模式：

| 模式 | 用途 | API Key | 网络请求 |
|---|---|---|---|
| `mock`（默认） | 课堂演示、离线复现、自动检查 | 不需要 | 不发送 |
| `live` | 观察真实模型的工具选择 | 需要 `DEEPSEEK_API_KEY` | 发送到 DeepSeek API |

Live 模式使用 OpenAI Python SDK 连接 DeepSeek 兼容接口：

- 地址：`https://api.deepseek.com`
- 模型：`deepseek-flash`
- 能力：聊天 + Tool Calls

Notebook 不会把 API Key 写入文件或打印出来，但 Live 模式会把提示词和模拟工具结果发送给模型服务。不要在练习数据中放入真实订单、用户隐私或生产凭据。


In [1]:
# 默认 mock 模式不需要安装第三方包。
# 只有使用 Live 模式时，才需要在当前环境中安装经过验证的版本：
# python -m pip install "openai==3.3.1"

print("环境说明已加载：默认使用 mock 模式；Live 模式需要 openai==3.3.1。")


环境说明已加载：默认使用 mock 模式；Live 模式需要 openai==3.3.1。


In [2]:
import copy
import json
import os
from collections import Counter
from datetime import datetime, timezone
from getpass import getpass
from importlib.metadata import PackageNotFoundError, version
from types import SimpleNamespace
from uuid import uuid4

RUN_MODE = os.getenv("HARNESS_LAB_MODE", "mock").strip().lower()
if RUN_MODE not in {"mock", "live"}:
    raise ValueError("HARNESS_LAB_MODE 只能是 mock 或 live")

OpenAI = None
if RUN_MODE == "live":
    try:
        from openai import OpenAI
        installed_version = version("openai")
    except (ImportError, PackageNotFoundError) as exc:
        raise ImportError(
            'Live 模式需要先安装：python -m pip install "openai==3.3.1"'
        ) from exc
    if installed_version != "3.3.1":
        print(f"提示：本实验验证版本为 openai==3.3.1，当前为 {installed_version}。")


def show(data):
    """把可序列化对象打印成缩进的中文 JSON，方便阅读。"""
    print(json.dumps(data, ensure_ascii=False, indent=2))


print(f"运行模式：{RUN_MODE}")


运行模式：mock


### 选择运行模式

默认直接运行即可，使用确定性的离线模拟，不产生 API 费用。

如需观察真实模型行为，请在启动 Jupyter 前设置：

```powershell
# Windows PowerShell
$env:HARNESS_LAB_MODE="live"
$env:DEEPSEEK_API_KEY="你的 DeepSeek API Key"
jupyter lab
```

```bash
# macOS / Linux
export HARNESS_LAB_MODE=live
export DEEPSEEK_API_KEY="你的 DeepSeek API Key"
jupyter lab
```

不要把真实 Key 写进代码、输出或 Git。共享已经运行过的 Notebook 前，应再次检查输出中是否含有本机路径、凭据或业务数据。


In [3]:
BASE_URL = "https://api.deepseek.com"
MODEL = "deepseek-flash"
client = None

if RUN_MODE == "live":
    api_key = os.getenv("DEEPSEEK_API_KEY")
    if not api_key:
        api_key = getpass("请输入 DeepSeek API Key（输入不会显示）：")
    if not api_key:
        raise ValueError("没有读取到 DEEPSEEK_API_KEY")
    client = OpenAI(api_key=api_key, base_url=BASE_URL)
    print(f"Live 客户端已创建：{BASE_URL} / {MODEL}")
    print("API Key 已读取，内容不会显示。")
else:
    print("Mock 模式已就绪：不会读取 API Key，也不会发送网络请求。")


Mock 模式已就绪：不会读取 API Key，也不会发送网络请求。


### 先确认：现在只会“聊天”，还不是 Agent

下面只发一轮文字，没有工具。Mock 模式返回固定教学答案；Live 模式用于确认 SDK、Key 和模型是否连通。


In [5]:
if RUN_MODE == "live":
    hello = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "你是一名简洁的助教，用一句话回答。"},
            {"role": "user", "content": "用一句话说明：大模型为什么不能自己去查公司数据库？"},
        ],
        max_tokens=200,
        extra_body={"thinking": {"type": "disabled"}},
    )
    print(hello.choices[0].message.content)
    print("实际模型：", hello.model)
else:
    print("大模型本身没有公司数据库的接口、权限和实时数据，需要由外部程序通过受控工具取得事实。")
    print("实际模型：mock-teaching-model")


大模型本身没有公司数据库的接口、权限和实时数据，需要由外部程序通过受控工具取得事实。
实际模型：mock-teaching-model


# 第一部分　最小 Agent Loop

对应课件 03–08 页。

## 聊天机器人和 Agent 差在哪？

| | 聊天机器人 | Agent |
|---|---|---|
| 做一次还是做多步 | 你问一句，它答一句，结束 | 朝着一个目标，可能走很多步 |
| 有没有手 | 只能根据已有文字猜 | 可以申请调用工具，拿到真实结果 |
| 下一步谁决定 | 等你再问 | 看上一工具返回了什么，再决定 |

**关键句：** Agent 的关键不是“说得更长”，而是**工具返回的新事实会改变下一步**。


## 1.1 用一个很小的任务看循环

任务只有一句话：

> 检查订单 A1024 为什么退款失败，并给出下一步建议。

如果这是聊天机器人，它只能猜。  
如果这是 Agent，它应该去查，而且每查一步都会产生**新问题**：

| 它做了什么 | 看到什么 | 因此下一步想问什么 |
|---|---|---|
| 读订单 | 退款状态 = 失败 | 那支付侧呢？ |
| 查支付 | 原交易还在处理中 | 这种状态允许退款吗？ |
| 核对规则 | 处理中不可退款 | 可以给原因和建议了 |

现在先只观察一件事：**每得到一个新事实，下一步要查的问题就更具体。**等这个循环真正跑完后，我们再讨论它和固定步骤程序有什么区别。


## 1.2 先自己调用工具，不要一上来就交给模型

工具不是魔法，就是普通 Python 函数。  
下面三个函数返回的是**演练数据**（假数据），真实项目里可以换成查数据库、查支付接口。


In [6]:
def order_get(order_id="A1024"):
    """读订单。这一步故意只告诉你“退款失败”，不告诉支付侧原因。"""
    return {
        "order_id": order_id,
        "refund_status": "失败",
        "amount": "88.00",
        "note": "只知道退款失败，支付侧状态未知",
    }


def payment_query(order_id="A1024"):
    """查原交易。这里才出现关键新事实：还在处理中。"""
    return {
        "order_id": order_id,
        "original_txn_status": "处理中",
        "channel": "BankPay",
    }


def policy_lookup(scene="refund_while_processing"):
    """查退款规则，给出原因和建议。"""
    return {
        "scene": scene,
        "rule": "原交易处理中不可退款",
        "advice": "等待交易到达终态后，再重试退款",
    }


print("【你自己调用】读订单：")
show(order_get())
print("\n【你自己调用】查支付：")
show(payment_query())
print("\n【你自己调用】查规则：")
show(policy_lookup())


【你自己调用】读订单：
{
  "order_id": "A1024",
  "refund_status": "失败",
  "amount": "88.00",
  "note": "只知道退款失败，支付侧状态未知"
}

【你自己调用】查支付：
{
  "order_id": "A1024",
  "original_txn_status": "处理中",
  "channel": "BankPay"
}

【你自己调用】查规则：
{
  "scene": "refund_while_processing",
  "rule": "原交易处理中不可退款",
  "advice": "等待交易到达终态后，再重试退款"
}


现在我们作为开发者已经知道标准答案：原交易还在处理中，规则不允许这时退款。

先记住这条证据链。下一节我们故意不把工具结果给模型，看它单靠语言能力能不能知道这些事实；然后再一步步把工具接给它。


## 1.3 把工具结果拿走：模型只能承认不知道，或者猜

下面把同样的任务直接丢给模型，**不提供任何工具**。  
它可能会说得头头是道，但那些内容不是查来的，是编/猜的。


In [6]:
if RUN_MODE == "live":
    guess = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "你是退款助手。只能根据已有文字回答，不要假装查过系统。"},
            {"role": "user", "content": "检查订单 A1024 为什么退款失败，并给出下一步建议。"},
        ],
        max_tokens=400,
        extra_body={"thinking": {"type": "disabled"}},
    )
    guess_text = guess.choices[0].message.content
else:
    guess_text = (
        "我没有订单、支付状态或退款规则的查询结果，因此不能确认失败原因。"
        "请先查询订单和原支付交易，再依据规则给出建议。"
    )

print("没有工具时，模型只能这样答：")
print(guess_text)
print("\n注意：它没有 order_get / payment_query 的返回值，所以无法引用‘处理中不可退款’这条事实。")


没有工具时，模型只能这样答：
我没有订单、支付状态或退款规则的查询结果，因此不能确认失败原因。请先查询订单和原支付交易，再依据规则给出建议。

注意：它没有 order_get / payment_query 的返回值，所以无法引用‘处理中不可退款’这条事实。


## 1.4 工具有两张脸：给模型看的说明书，给程序用的函数

现在的问题是：三个 Python 函数虽然已经存在，模型却不知道它们叫什么、什么时候用、参数怎样写。

模型**看不到** Python 源码，也**不会**自己执行函数。我们需要同时准备给模型看的 Schema 和给程序执行的 Python 函数：

![Tool Calls 从模型申请到程序执行再返回 Context 的往返过程](https://zhangyeio.github.io/assets/images/deepseek-harness-lab/02-function-calling-roundtrip.svg)

当前 DeepSeek API 使用 **Tool Calls** 这一术语；`function` 是工具类型。很多资料仍把这套机制称为 Function Calling，本实验在概念说明中把两者视为同一条调用链。

这一步只解决了**模型怎样表达“我想调用工具”**，还没有解决怎样执行、怎样把结果交还给模型。下一节的 Loop 才负责把这条链路闭合。


In [7]:
def make_schema(name, description, properties):
    """生成工具 Schema；所有声明的参数都要求显式提供，并拒绝未知字段。"""
    return {
        "type": "function",
        "function": {
            "name": name,
            "description": description,
            "parameters": {
                "type": "object",
                "properties": properties,
                "required": list(properties),
                "additionalProperties": False,
            },
        },
    }


REFUND_TOOLS = {
    "order_get": order_get,
    "payment_query": payment_query,
    "policy_lookup": policy_lookup,
}

REFUND_SCHEMAS = [
    make_schema("order_get", "读取订单的退款状态和金额。", {
        "order_id": {"type": "string", "description": "订单号，例如 A1024"},
    }),
    make_schema("payment_query", "查询该订单对应的原支付交易状态。", {
        "order_id": {"type": "string", "description": "订单号"},
    }),
    make_schema("policy_lookup", "核对退款规则，判断当前状态是否允许退款。", {
        "scene": {"type": "string", "description": "规则场景，例如 refund_while_processing"},
    }),
]

print("程序侧注册了：", list(REFUND_TOOLS))
print("\n模型将看到的第一份说明书：")
show(REFUND_SCHEMAS[0])


程序侧注册了： ['order_get', 'payment_query', 'policy_lookup']

模型将看到的第一份说明书：
{
  "type": "function",
  "function": {
    "name": "order_get",
    "description": "读取订单的退款状态和金额。",
    "parameters": {
      "type": "object",
      "properties": {
        "order_id": {
          "type": "string",
          "description": "订单号，例如 A1024"
        }
      },
      "required": [
        "order_id"
      ],
      "additionalProperties": false
    }
  }
}


## 1.5 从 Tool Calls 到 Agent Loop

Tool Calls 只产生工具调用申请；只有程序验证参数、执行工具、把结果写回 Context，并让新事实影响下一步，才形成 Agent Loop。

![Agent Loop 中模型、工具执行与 Context 更新的循环](https://zhangyeio.github.io/assets/images/deepseek-harness-lab/03-agent-loop.svg)

下面不直接扔出完整循环，而是分两步实现。第一步先封装两个动作：

- `ask_model(...)`：把当前 Context 和工具说明书发给模型；
- `execute_tool(...)`：根据模型点出的名字，验证参数并真正执行 Python 函数。

先把这两个动作看懂，再在下一格把它们放进循环。


In [8]:
PYTHON_TYPES = {
    "string": str,
    "number": (int, float),
    "integer": int,
    "boolean": bool,
    "array": list,
    "object": dict,
}


def schema_index(schemas):
    return {item["function"]["name"]: item for item in schemas}


def validate_tool_arguments(schema, arguments):
    """应用侧参数校验：不要把模型生成的 JSON 直接传给业务函数。"""
    if not isinstance(arguments, dict):
        return "工具参数必须是 JSON 对象"

    parameters = schema["function"]["parameters"]
    properties = parameters.get("properties", {})
    required = parameters.get("required", [])

    missing = [name for name in required if name not in arguments]
    if missing:
        return "缺少必填参数：" + "、".join(missing)

    unknown = [name for name in arguments if name not in properties]
    if unknown and parameters.get("additionalProperties") is False:
        return "包含未声明参数：" + "、".join(unknown)

    for name, value in arguments.items():
        expected_name = properties.get(name, {}).get("type")
        expected_type = PYTHON_TYPES.get(expected_name)
        if expected_type and not isinstance(value, expected_type):
            return f"参数 {name} 应为 {expected_name}，实际为 {type(value).__name__}"
        if expected_name in {"integer", "number"} and isinstance(value, bool):
            return f"参数 {name} 应为 integer，不能是 boolean"
    return None


def execute_tool(registry, name, arguments, schemas):
    """模型只负责点名；应用先校验，再真正调用 Python 函数。"""
    if name not in registry:
        return {"error": f"没有这个工具：{name}"}

    selected_schema = schema_index(schemas).get(name)
    if selected_schema is None:
        return {"error": f"没有这个工具的 Schema：{name}"}

    validation_error = validate_tool_arguments(selected_schema, arguments)
    if validation_error:
        return {"error": "invalid_arguments", "message": validation_error}

    try:
        return registry[name](**arguments)
    except Exception as exc:
        return {"error": type(exc).__name__, "message": str(exc)}


class MockMessage:
    """只实现教学循环会用到的消息字段。"""

    def __init__(self, content=None, tool_calls=None):
        self.content = content
        self.tool_calls = tool_calls

    def model_dump(self, exclude_none=True):
        data = {"role": "assistant", "content": self.content}
        if self.tool_calls:
            data["tool_calls"] = [
                {
                    "id": call.id,
                    "type": "function",
                    "function": {
                        "name": call.function.name,
                        "arguments": call.function.arguments,
                    },
                }
                for call in self.tool_calls
            ]
        if exclude_none:
            data = {key: value for key, value in data.items() if value is not None}
        return data


MOCK_ARGUMENTS = {
    "order_get": {"order_id": "A1024"},
    "payment_query": {"order_id": "A1024"},
    "policy_lookup": {"scene": "refund_while_processing"},
    "query_metrics": {"metric": "payment_success_rate", "window": "30m"},
    "query_traces": {"service": "route-service"},
    "query_success_by_slice": {"dimension": "version_x_route"},
    "compare_retry_config": {"old_version": "v2.3.6", "new_version": "v2.3.7"},
    "check_vendor_status": {"vendor": "third_party_pay"},
    "verify_rollback": {"action": "rollback_v2.3.7_route_config"},
}


def called_tool_names(messages):
    names = []
    for item in messages:
        if not isinstance(item, dict):
            continue
        for call in item.get("tool_calls") or []:
            names.append(call["function"]["name"])
    return names


def mock_final_answer(available_names):
    names = set(available_names)
    if {"order_get", "payment_query", "policy_lookup"}.issubset(names):
        return (
            "原因：订单 A1024 的退款状态为失败，而原支付交易仍处于处理中；规则明确规定"
            "原交易处理中不可退款。[证据: order_get][证据: payment_query][证据: policy_lookup]\n\n"
            "建议：等待原交易到达终态后再重试退款。现有工具没有提供具体等待时长，因此不补充时长承诺。"
        )
    if {
        "query_metrics", "query_traces", "query_success_by_slice",
        "compare_retry_config", "check_vendor_status", "verify_rollback",
    }.issubset(names):
        return (
            "现象：10:20–10:42 支付成功率从 98.7% 降至 91.4%。[证据: query_metrics]\n"
            "失败环节：请求在已限流的兼容端点超时，重试进一步放大流量。[证据: query_traces]\n"
            "范围：仅 v2.3.7 与新路由组合显著异常。[证据: query_success_by_slice]\n"
            "被排除原因：第三方状态页和备用渠道正常，不支持全局通道故障假设。[证据: check_vendor_status]\n"
            "根因：v2.3.7 新路由加载了错误的重试目标，把请求指向已限流端点。"
            "[证据: compare_retry_config]\n"
            "验证与建议：回滚后成功率恢复到约 98%，应保留回滚、限制重试并补充灰度验证。"
            "[证据: verify_rollback]"
        )
    return "现有证据不足，不能提交结论。"


def ask_model(messages, schemas):
    """Live 模式请求 DeepSeek；Mock 模式按固定顺序返回可复现的工具申请。"""
    if RUN_MODE == "live":
        kwargs = {
            "model": MODEL,
            "messages": messages,
            "max_tokens": 800,
            "extra_body": {"thinking": {"type": "disabled"}},
        }
        if schemas:
            kwargs["tools"] = schemas
            kwargs["tool_choice"] = "auto"
        return client.chat.completions.create(**kwargs).choices[0].message

    available_names = [item["function"]["name"] for item in schemas]
    already_called = set(called_tool_names(messages))
    missing = [name for name in available_names if name not in already_called]
    if missing:
        name = missing[0]
        arguments = MOCK_ARGUMENTS[name]
        tool_call = SimpleNamespace(
            id=f"call_mock_{name}",
            function=SimpleNamespace(
                name=name,
                arguments=json.dumps(arguments, ensure_ascii=False),
            ),
        )
        return MockMessage(tool_calls=[tool_call])
    return MockMessage(content=mock_final_answer(available_names))


### 先停在一次 Tool Call，不急着循环

下面只做一轮，并且暂时只把 `order_get` 这一个工具告诉模型。这样可以单独观察三件事：

1. 模型返回的是工具名和参数，不是订单数据；
2. 应用校验参数并执行后，我们才拿到真实结果；
3. 此时程序还没有把结果发回模型，所以模型也还不能继续分析。

先看懂这一轮，再让程序自动重复它。


In [9]:
one_step_messages = [
    {"role": "system", "content": "你是退款助手。先调用提供的工具读取订单，不要直接猜原因。"},
    {"role": "user", "content": "检查订单 A1024 为什么退款失败。"},
]

one_reply = ask_model(one_step_messages, [REFUND_SCHEMAS[0]])

if one_reply.tool_calls:
    one_call = one_reply.tool_calls[0]
    one_name = one_call.function.name
    one_args = json.loads(one_call.function.arguments or "{}")
    print("模型申请：", one_name, one_args)

    one_result = execute_tool(REFUND_TOOLS, one_name, one_args, [REFUND_SCHEMAS[0]])
    print("应用校验并执行后才得到：")
    show(one_result)
    print("\n现在先停下：结果还没有写回 messages，模型还没看到它。")
else:
    print("模型这次没有申请工具；Live 模式下可重新运行本格观察。")


模型申请： order_get {'order_id': 'A1024'}
应用校验并执行后才得到：
{
  "order_id": "A1024",
  "refund_status": "失败",
  "amount": "88.00",
  "note": "只知道退款失败，支付侧状态未知"
}

现在先停下：结果还没有写回 messages，模型还没看到它。


### 第二步：把两个动作串成循环

上一格停在“程序已经拿到订单数据，但模型还没看到”。要让模型继续，就必须把工具结果追加到 `messages`，然后再问一轮。

与其每次手动重复，我们加一个 `for` 循环。每轮按同样的顺序走：

```text
① 问模型
② 保存模型回复
③ 如果有工具申请：执行并写回结果，然后继续下一轮
④ 如果只有文字：返回答案，结束循环
```

读代码时跟着①②③④找，不需要逐行记忆 API 字段。


In [10]:
def run_basic_loop(task, registry, schemas, system_prompt, max_steps=8):
    """最小循环：问模型 → 校验并执行工具 → 写回结果 → 再问。"""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": task},
    ]

    for step in range(1, max_steps + 1):
        print(f"\n========== 第 {step} 轮 ==========")
        message = ask_model(messages, schemas)
        messages.append(message.model_dump(exclude_none=True))

        if message.tool_calls:
            for call in message.tool_calls:
                name = call.function.name
                try:
                    args = json.loads(call.function.arguments or "{}")
                except json.JSONDecodeError as exc:
                    args = {}
                    result = {"error": "invalid_json", "message": str(exc)}
                else:
                    result = execute_tool(registry, name, args, schemas)

                print(f"模型申请调用：{name}")
                print("参数：", args)
                print("应用执行后返回：")
                show(result)

                messages.append({
                    "role": "tool",
                    "tool_call_id": call.id,
                    "content": json.dumps(result, ensure_ascii=False),
                })
            continue

        print("模型给出文字答案：")
        print(message.content)
        return {"answer": message.content, "messages": messages, "steps": step}

    return {"answer": None, "messages": messages, "steps": max_steps}


## 1.6 跑一遍退款任务

看输出时先不纠结模型一轮申请了几个工具，只抓住一件事：**前一轮的工具返回，有没有改变后一轮要查什么。**

这里把一轮理解成一次模型请求；一次请求里可能包含一个或多个工具调用。


In [11]:
REFUND_PROMPT = """
你是退款查询助手。请使用工具查找原因，再给建议。
根据已有证据选择下一步工具；已经成功返回的不要重复调用。
证据足够时，用中文写出：原因 + 建议。
关键事实后必须标注 [证据: 工具名]；不得补充工具未返回的等待时长、清算周期或业务承诺。
""".strip()

refund_run = run_basic_loop(
    task="检查订单 A1024 为什么退款失败，并给出下一步建议。",
    registry=REFUND_TOOLS,
    schemas=REFUND_SCHEMAS,
    system_prompt=REFUND_PROMPT,
)

print("\n一共走了几轮：", refund_run["steps"])



========== 第 1 轮 ==========
模型申请调用：order_get
参数： {'order_id': 'A1024'}
应用执行后返回：
{
  "order_id": "A1024",
  "refund_status": "失败",
  "amount": "88.00",
  "note": "只知道退款失败，支付侧状态未知"
}

========== 第 2 轮 ==========
模型申请调用：payment_query
参数： {'order_id': 'A1024'}
应用执行后返回：
{
  "order_id": "A1024",
  "original_txn_status": "处理中",
  "channel": "BankPay"
}

========== 第 3 轮 ==========
模型申请调用：policy_lookup
参数： {'scene': 'refund_while_processing'}
应用执行后返回：
{
  "scene": "refund_while_processing",
  "rule": "原交易处理中不可退款",
  "advice": "等待交易到达终态后，再重试退款"
}

========== 第 4 轮 ==========
模型给出文字答案：
原因：订单 A1024 的退款状态为失败，而原支付交易仍处于处理中；规则明确规定原交易处理中不可退款。[证据: order_get][证据: payment_query][证据: policy_lookup]

建议：等待原交易到达终态后再重试退款。现有工具没有提供具体等待时长，因此不补充时长承诺。

一共走了几轮： 4


## 1.7 第一部分检查点：它已经是 Agent，但还只是 Demo

Agent 和 Workflow 不是非黑即白。确定性 Workflow 也可以分支，Agent 也通常被嵌入更大的工作流；这里比较的是“步骤主要由预先编排决定”与“下一步主要由模型根据结果选择”两种控制方式。

![确定性工作流与模型驱动 Agent 的控制方式对比](https://zhangyeio.github.io/assets/images/deepseek-harness-lab/04-agent-vs-workflow.svg)

在退款案例中，支付查询如果返回“交易已成功”，模型就应根据这个新事实选择另一条调查路径。

但当前裸 Loop 还有一个致命弱点：只要模型没有返回 `tool_calls`，代码就直接 `return`。假设它只查了订单便回答：

> “退款失败可能是银行繁忙，建议稍后重试。”

支付状态和退款规则都没查，裸 Loop 仍然会接受，因为它只判断**模型想不想结束**，没有判断**任务是不是真的完成**。

到这里先看它还缺哪些系统能力：

| 裸 Loop 的问题 | 需要补上的能力 |
|---|---|
| 工具调用散落在代码里 | 统一管理、校验和执行工具 |
| 跑完后难以回放 | 按每次运行记录真实执行过程 |
| 模型想结束就结束 | 检查证据是否完整且有效 |
| 只能靠人眼看答案 | 按标准自动验收 |

第二部分会逐个给这四种能力命名并实现。Harness 不是突然出现的新魔法，只是把裸 Loop 缺少的工程能力补齐。


# 第二部分　给循环加上 Harness

对应课件 09–24 页。

## 2.1 一个类比：聪明新员工也需要工位

把模型当成新员工：

| 新员工需要什么 | Harness 对应什么 |
|---|---|
| 桌上的资料、当前任务 | Context（由 Session 维护） |
| 哪些系统账号能登录 | Tools |
| 做一步、看结果、再做一步 | Loop |
| 主管在关键节点检查 | Hook |
| 下班前的交付验收 | Evaluation |

Harness **不替员工思考**，只提供能把工作做完的环境。第一部分的裸 Loop 加上这些工程能力，就构成本课的最小 Harness：

![教学版 Harness 的 Session、Tools、Loop、Hook 与 Evaluation 组件](https://zhangyeio.github.io/assets/images/deepseek-harness-lab/05-harness-components.svg)

接下来每增加一个组件，都只问一个问题：**它在补裸 Loop 的哪个缺口？**


## 2.2 为什么不能只靠 Prompt？

三件事经常被混在一起：

| | 它管什么 | 它不管什么 | 退款例子 |
|---|---|---|---|
| **Prompt** | 这一次用什么语气、什么原则 | 工具会不会真的执行、能不能停 | “请先查订单再查支付” |
| **Function Calling** | 模型用 JSON 申请调用哪个函数 | 函数执不执行、超时、权限、完成标准 | 输出 `order_get(...)` |
| **Harness** | 多轮怎么串起来：执行、记账、拦截、验收 | 不替模型思考 | 查完 3 个工具才允许结束 |

同一句要求，放在不同位置，强度完全不同：

```text
Prompt：请务必查完支付状态和退款规则再回答。
         → 模型通常会遵守，但仍可能漏查。

Hook：required = {payment_query, policy_lookup}
       → Session 中没有成功记录，就一定返回“不允许结束”。
```

写操作也是一样：Prompt 中写“退款超过 1000 元请谨慎”只是提醒；Tools 层写 `if amount > 1000: require_approval()` 才是真正的系统控制。

> **写进 Prompt ≠ 系统具备这个能力。**

真正的权限、执行和停止条件必须由代码实现，这正是 Harness 的职责。下面从一个极小的插件盒子开始，把这些能力逐个装上。


## 2.3 先放一个极小的“插件盒子”

接下来会依次写 Session、Tools、Hook、Loop 和 Evaluation。为了让它们能互相找到，我们先用一个字典把它们装起来。

这里先不要学习框架设计，也不需要背 `MiniKernel`。它不包含任何 Agent 业务逻辑，只做两件事：

1. `mount(名字, 插件)`：插上一个零件
2. `get(名字)`：按名字把零件拿出来用

后面会依次插上：`session`、`tools`、`hook`、`loop`、`eval`。

可以把它当成插线板。真正需要理解的是：每个零件解决了上一部分的哪个问题。


In [12]:
class MiniKernel:
    """很小的运行时：只负责保管插件。"""

    def __init__(self):
        self.plugins = {}  # 名字 → 插件对象

    def mount(self, name, plugin):
        plugin.kernel = self  # 让插件能找到内核，从而找到其他插件
        self.plugins[name] = plugin
        print(f"已挂载插件：{name}")
        return plugin

    def get(self, name):
        return self.plugins[name]


kernel = MiniKernel()
print("现在还是空的：", list(kernel.plugins))


现在还是空的： []


## 2.4 Session：模型的工作记忆，不等于系统的流水账

第一部分已经有 `messages`，为什么还需要 Session？

![模型消息 Context 与系统可信 Session 流水账的区别](https://zhangyeio.github.io/assets/images/deepseek-harness-lab/06-messages-vs-session.svg)

阅读下面的代码时，先只关注两个地方：

- `events`：保存系统真正执行过的事件；
- `record(...)`：每次工具执行后追加一条记录。

`successful_tools()` 和 `replay()` 只是后面门禁与回放要用的查询方法，不需要现在记住实现细节。

有了可信流水账，下一步就让所有工具执行都自动往里面记一笔，所以接下来实现 Tools。


In [13]:
class SessionPlugin:
    """当前运行的可信流水账；每次 run 都创建独立 run_id 并清空旧事件。"""

    def __init__(self):
        self.run_id = None
        self.events = []

    def start_run(self, task):
        self.run_id = uuid4().hex
        self.events = []
        self.record("user_message", task=task)
        return self.run_id

    def record(self, event_type, **info):
        if self.run_id is None:
            raise RuntimeError("请先调用 start_run() 创建独立运行")
        event = {
            "type": event_type,
            "run_id": self.run_id,
            "timestamp": datetime.now(timezone.utc).isoformat(),
            **copy.deepcopy(info),
        }
        self.events.append(event)
        return event

    def successful_results(self, name=None):
        results = [
            event for event in self.events
            if event.get("run_id") == self.run_id
            and event.get("type") == "tool/result"
            and event.get("ok")
        ]
        if name is not None:
            results = [event for event in results if event.get("name") == name]
        return results

    def successful_tools(self):
        return {event["name"] for event in self.successful_results()}

    def replay(self):
        if not self.events:
            print("（还没有任何记录）")
            return
        for event in self.events:
            kind = event.get("type")
            step = event.get("step", "-")
            if kind == "user_message":
                print("  新运行已创建")
            elif kind == "tool/call":
                print(f"  第 {step} 轮  申请调用 {event['name']}  {event.get('arguments')}")
            elif kind == "tool/result":
                print(f"  第 {step} 轮  工具返回 {event['name']}  成功={event.get('ok')}")
            elif kind == "hook/block":
                print(f"  第 {step} 轮  还不能结束：{event.get('feedback')}")
            elif kind == "hook/pass":
                print(f"  第 {step} 轮  完成检查通过")
            elif kind == "turn/end":
                print(f"  第 {step} 轮  任务结束，原因：{event.get('reason')}")


kernel.mount("session", SessionPlugin())
demo_session = kernel.get("session")
demo_session.start_run("Session 演示")
print("Session 已就绪：每次新任务都会获得独立 run_id。")


已挂载插件：session
Session 已就绪：每次新任务都会获得独立 run_id。


## 2.5 Tools：把“模型想做”变成“系统允许并执行”

第一部分里，`execute_tool` 是一个散落的函数。现在把它收进插件，让所有工具都经过同一个入口：

```text
模型申请 → Schema 与参数校验 → 能不能做 → 真正执行 → 记录结果
```

工具也有不同风险等级：

| 工具例子 | 类型 | Tools 层可以增加的控制 |
|---|---|---|
| `order_get()` 查询订单 | 只读 | 参数校验、超时、访问权限 |
| `refund_create()` 发起退款 | 写操作 | 金额上限、人工审批、幂等检查、审计日志 |

模型对两者都只是在“申请调用”，真正决定是否允许执行的应该是 Tools 层。

这样以后增加权限、参数校验、超时、审批时，不需要修改 Agent 的思考逻辑，只需要增强 Tools 层。

现在 Session 已经能回答“本次运行实际做过什么”。有了这份可信记录，我们才能继续回答下一个问题：模型提交答案时，证据够不够？这就引出 Hook。


In [14]:
class ToolPlugin:
    """工具登记处：负责 Schema、参数校验、执行和记账。"""

    def __init__(self):
        self.functions = {}
        self.schemas = []
        self.schema_by_name = {}

    def clear(self):
        self.functions.clear()
        self.schemas.clear()
        self.schema_by_name.clear()

    def register(self, name, function, schema):
        if schema["function"]["name"] != name:
            raise ValueError("工具名与 Schema 名不一致")
        self.functions[name] = function
        self.schema_by_name[name] = schema
        self.schemas = [
            item for item in self.schemas
            if item["function"]["name"] != name
        ]
        self.schemas.append(schema)
        print(f"  已登记工具：{name}")

    def execute(self, name, arguments, step):
        session = self.kernel.get("session")
        session.record("tool/call", step=step, name=name, arguments=arguments)

        if name not in self.functions:
            result = {"error": f"未登记工具：{name}"}
            session.record(
                "tool/result", step=step, name=name, arguments=arguments,
                ok=False, result=result,
            )
            return False, result

        validation_error = validate_tool_arguments(self.schema_by_name[name], arguments)
        if validation_error:
            result = {"error": "invalid_arguments", "message": validation_error}
            session.record(
                "tool/result", step=step, name=name, arguments=arguments,
                ok=False, result=result,
            )
            return False, result

        try:
            # 业务数据可以合法包含名为 error 的字段，例如 PAYMENT_TIMEOUT。
            # 是否成功应由执行是否抛出异常决定，不能根据任意返回字段名猜测。
            result = self.functions[name](**arguments)
            ok = True
        except Exception as exc:
            result = {"error": type(exc).__name__, "message": str(exc)}
            ok = False

        session.record(
            "tool/result", step=step, name=name, arguments=arguments,
            ok=ok, result=result,
        )
        return ok, result


kernel.mount("tools", ToolPlugin())


已挂载插件：tools


## 2.6 Hook：先体验一次“模型想交卷，系统不允许”

假设退款任务里，模型只调用了 `order_get` 就回答“可能是银行问题，建议稍后重试”。  
听起来像那么回事，但支付状态和规则都没查。

Hook 要做的事非常具体：

1. 看当前 run_id 的 Session 中，工具、关键参数和返回字段是否都符合要求；
2. 和“必须完成的清单”对比；
3. 缺了就**不允许结束**，并把“还缺什么”写回对话，让模型继续。

```text
模型：我想提交答案。
Hook：先看流水账。你只查了订单，还缺支付状态和退款规则，不能交卷。
模型：收到缺口，继续调用工具。
```

下面用确定性数据制造一次“提前交卷”：Session 里只放入成功的 `order_get`，Hook 对照三个必查项，应该明确返回 `False` 并指出缺口。

> 为了便于入门，本教学版已经检查工具名、关键参数和关键返回字段；生产系统还应检查数据新鲜度、权限范围、来源可信度和证据之间是否相互支持。

Hook 已经会判断“能不能结束”。下一步把它接回第一部分的 Loop：模型给文字时不再直接返回，而是先过门禁。


In [15]:
class HookPlugin:
    """完成门禁：检查当前运行中的工具名、参数和关键返回字段。"""

    def __init__(self):
        self.required = {}

    @staticmethod
    def _matches(event, rule):
        expected_arguments = rule.get("arguments", {})
        actual_arguments = event.get("arguments", {})
        if any(actual_arguments.get(key) != value for key, value in expected_arguments.items()):
            return False

        expected_fields = rule.get("result_fields", {})
        actual_result = event.get("result", {})
        if not isinstance(actual_result, dict):
            return False
        return all(actual_result.get(key) == value for key, value in expected_fields.items())

    def check(self, step):
        session = self.kernel.get("session")
        missing = []
        for name, rule in self.required.items():
            candidates = session.successful_results(name)
            if not any(self._matches(event, rule) for event in candidates):
                missing.append(rule["label"])

        if missing:
            feedback = "还缺有效证据：" + "、".join(missing)
            session.record("hook/block", step=step, feedback=feedback)
            return False, feedback

        session.record("hook/pass", step=step, feedback="证据清单已齐，可以结束")
        return True, "证据清单已齐，可以结束"


kernel.mount("hook", HookPlugin())

demo_session = kernel.get("session")
demo_session.start_run("门禁演示")
demo_session.record(
    "tool/result", step=1, name="order_get", arguments={"order_id": "A1024"},
    ok=True, result={"order_id": "A1024", "refund_status": "失败"},
)
kernel.get("hook").required = {
    "order_get": {
        "label": "读取订单 A1024",
        "arguments": {"order_id": "A1024"},
        "result_fields": {"refund_status": "失败"},
    },
    "payment_query": {
        "label": "查询 A1024 支付状态",
        "arguments": {"order_id": "A1024"},
        "result_fields": {"original_txn_status": "处理中"},
    },
    "policy_lookup": {
        "label": "核对退款规则",
        "arguments": {"scene": "refund_while_processing"},
        "result_fields": {"rule": "原交易处理中不可退款"},
    },
}
allow, reason = kernel.get("hook").check(step=1)
print("只查了订单时，允许结束吗？", allow)
print("原因：", reason)


已挂载插件：hook
只查了订单时，允许结束吗？ False
原因： 还缺有效证据：查询 A1024 支付状态、核对退款规则


## 2.7 Loop：还是第一部分那个循环，只是改成向插件要东西

对照一下，逻辑没有变：

| 第一部分 | 现在 |
|---|---|
| `ask_model(messages, schemas)` | 还是它，schemas 来自 `tools.schemas` |
| `execute_tool(...)` | 换成 `tools.execute(...)` |
| 模型给文字就直接结束 | 先问 `hook.check()`，不通过就继续 |

循环的骨架没有变，多出来的是两个关键控制点：

- 工具执行后，由当前 run_id 的 Session 留下带参数和结果的可信记录；
- 模型给文字答案后，由 Hook 决定接受还是继续。

请特别关注下面代码中的 `hook.check(step)`：它把停止权从模型手里收回到了系统。

到这里运行中的控制已经完整；最后还差运行结束后的质量判断，因此下一节再加入 Evaluation。


In [16]:
class LoopPlugin:
    """把最小循环接到 Session / Tools / Hook 上。"""

    def __init__(self, max_steps=12):
        self.max_steps = max_steps
        self.system_prompt = "你是一名会使用工具的助手。"

    def run(self, task):
        tools = self.kernel.get("tools")
        hook = self.kernel.get("hook")
        session = self.kernel.get("session")
        run_id = session.start_run(task)

        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": task},
        ]

        for step in range(1, self.max_steps + 1):
            print(f"\n========== 第 {step} 轮 ==========")
            try:
                message = ask_model(messages, tools.schemas)
            except Exception as exc:
                session.record(
                    "turn/end", step=step, reason="model_error",
                    error={"type": type(exc).__name__, "message": str(exc)},
                )
                return {
                    "status": "model_error", "answer": None,
                    "steps": step, "run_id": run_id,
                }

            messages.append(message.model_dump(exclude_none=True))

            if message.tool_calls:
                for call in message.tool_calls:
                    name = call.function.name
                    try:
                        args = json.loads(call.function.arguments or "{}")
                    except json.JSONDecodeError as exc:
                        args = {}
                        ok = False
                        result = {"error": "invalid_json", "message": str(exc)}
                        session.record(
                            "tool/call", step=step, name=name,
                            arguments_text=call.function.arguments,
                        )
                        session.record(
                            "tool/result", step=step, name=name,
                            arguments=args, ok=ok, result=result,
                        )
                    else:
                        ok, result = tools.execute(name, args, step)

                    print(f"模型申请调用：{name}")
                    print("参数：", args)
                    print("Harness 执行结果：")
                    show(result)
                    messages.append({
                        "role": "tool",
                        "tool_call_id": call.id,
                        "content": json.dumps(result, ensure_ascii=False),
                    })
                continue

            print("模型想提交最终答案：")
            print(message.content)
            allow, feedback = hook.check(step)
            if not allow:
                print("门禁未通过：", feedback)
                messages.append({
                    "role": "user",
                    "content": (
                        "Harness 完成检查：你还不能结束。" + feedback
                        + "。请调用还缺的工具，不要重复已经成功的调用。"
                    ),
                })
                continue

            print("门禁通过，任务结束。")
            session.record("turn/end", step=step, reason="completed")
            return {
                "status": "completed", "answer": message.content,
                "steps": step, "run_id": run_id,
            }

        session.record("turn/end", step=self.max_steps, reason="max_steps")
        return {
            "status": "max_steps", "answer": None,
            "steps": self.max_steps, "run_id": run_id,
        }


kernel.mount("loop", LoopPlugin())


已挂载插件：loop


## 2.8 Evaluation：Hook 管交卷，Evaluation 管评分

![Hook 在运行时控制是否交卷，Evaluation 在运行后评估过程与结果](https://zhangyeio.github.io/assets/images/deepseek-harness-lab/07-hook-vs-evaluation.svg)

下面的教学版评分器会把过程检查和结果检查合并成一个百分制报告。


In [17]:
class EvalPlugin:
    """把一组对/错检查收成百分制；空检查集不会发生除零错误。"""

    def evaluate(self, process_checks, result_checks):
        report = {**process_checks, **result_checks}
        if not report:
            return {"score": 0, "checks": {}, "warning": "没有提供任何检查项"}
        score = round(100 * sum(bool(value) for value in report.values()) / len(report))
        return {"score": score, "checks": report}


kernel.mount("eval", EvalPlugin())
print("当前已挂载：", list(kernel.plugins))


已挂载插件：eval
当前已挂载： ['session', 'tools', 'hook', 'loop', 'eval']


## 2.9 先用 Harness 把退款任务再跑一遍

不要一上来就换复杂案例。先用同一道退款题验证：换成插件版循环后，必须 3 个工具都成功，系统才允许结束。

运行时观察两件事：

1. 工具结果仍然通过 `role=tool` 回到模型，Agent Loop 没有变；
2. 模型提交文字答案时，Hook 会读取 Session。证据齐就通过，证据不齐就反馈缺口并继续。

Mock 模式行为固定，便于课堂复现；Live 模式的模型行为有随机性，可能主动一次查齐，也可能提前交卷。无论模型怎样选择，系统的完成标准都不会改变。上一节的确定性演示已经证明门禁会拒绝缺少证据的答案。


In [18]:
tools = kernel.get("tools")
tools.clear()
for name, fn, schema in zip(REFUND_TOOLS.keys(), REFUND_TOOLS.values(), REFUND_SCHEMAS):
    tools.register(name, fn, schema)

kernel.get("hook").required = {
    "order_get": {
        "label": "读取订单 A1024",
        "arguments": {"order_id": "A1024"},
        "result_fields": {"order_id": "A1024", "refund_status": "失败"},
    },
    "payment_query": {
        "label": "查询 A1024 支付状态",
        "arguments": {"order_id": "A1024"},
        "result_fields": {"order_id": "A1024", "original_txn_status": "处理中"},
    },
    "policy_lookup": {
        "label": "核对处理中退款规则",
        "arguments": {"scene": "refund_while_processing"},
        "result_fields": {"rule": "原交易处理中不可退款"},
    },
}
kernel.get("loop").system_prompt = REFUND_PROMPT

refund_harness = kernel.get("loop").run(
    "检查订单 A1024 为什么退款失败，并给出下一步建议。"
)

print("\n状态：", refund_harness["status"])
print("答案：\n", refund_harness["answer"])
print("\n流水账回放：")
kernel.get("session").replay()


  已登记工具：order_get
  已登记工具：payment_query
  已登记工具：policy_lookup

========== 第 1 轮 ==========
模型申请调用：order_get
参数： {'order_id': 'A1024'}
Harness 执行结果：
{
  "order_id": "A1024",
  "refund_status": "失败",
  "amount": "88.00",
  "note": "只知道退款失败，支付侧状态未知"
}

========== 第 2 轮 ==========
模型申请调用：payment_query
参数： {'order_id': 'A1024'}
Harness 执行结果：
{
  "order_id": "A1024",
  "original_txn_status": "处理中",
  "channel": "BankPay"
}

========== 第 3 轮 ==========
模型申请调用：policy_lookup
参数： {'scene': 'refund_while_processing'}
Harness 执行结果：
{
  "scene": "refund_while_processing",
  "rule": "原交易处理中不可退款",
  "advice": "等待交易到达终态后，再重试退款"
}

========== 第 4 轮 ==========
模型想提交最终答案：
原因：订单 A1024 的退款状态为失败，而原支付交易仍处于处理中；规则明确规定原交易处理中不可退款。[证据: order_get][证据: payment_query][证据: policy_lookup]

建议：等待原交易到达终态后再重试退款。现有工具没有提供具体等待时长，因此不补充时长承诺。
门禁通过，任务结束。

状态： completed
答案：
 原因：订单 A1024 的退款状态为失败，而原支付交易仍处于处理中；规则明确规定原交易处理中不可退款。[证据: order_get][证据: payment_query][证据: policy_lookup]

建议：等待原交易到达终态后再重试退款。现有工具没有提供具体等待时长，因此不补充时长承诺。

流水账回

到这里，本部分开头结构图中的五个组件都已经在退款任务上工作过了。

下一部分不改这套主链，只替换业务工具、完成条件和评估标准。


# 第三部分　支付排障（课件案例）

对应课件 25–35 页。

任务：

> 线上支付成功率突然下降了，帮我查一下原因。

数据全部是演练用的，不是真实事故。工具都是**只读查询**，没有改配置、没有回滚写操作。

这一部分不是再学一套新架构，而是验证一个关键结论：**业务场景变复杂时，Harness 主链仍然可以复用。**


## 3.1 先写“怎样才算查完”，再让它去查

如果不定完成标准，Agent 很容易在“好像跟发布有关”时就停；方向可能对，但证据还不足以证明根因。

![支付故障从影响确认、根因定位到回滚验证的六项证据链](https://zhangyeio.github.io/assets/images/deepseek-harness-lab/08-incident-evidence-chain.svg)

这张证据链就是后面 Hook 完成清单的来源。教学案例要求六类证据全部齐全，因此是一个约束较强的 Agent；模型仍然可以根据当前结果决定调用顺序。


## 3.2 三阶段证据链，六个只读查询函数

每个函数只回答上图中的一个问题，避免一个“大而全”的工具直接吐出所有结论。这样模型必须根据新事实逐步更新判断。

为了方便对照后续取证，先给出教师视角的标准答案：**v2.3.7 在新路由下加载了错误的重试配置**。知道答案不等于完成调查，Agent 仍要用证据证明它，并排除第三方故障等替代解释。

下面不把六个函数一次铺开，而是按证据链分成三格。

### 阶段一：先确认影响，再定位失败环节

第一格只定义 `query_metrics` 和 `query_traces`：一个回答“发生了什么”，一个回答“失败在哪一环”。


In [19]:
def query_metrics(metric="payment_success_rate", window="30m"):
    """1. 影响范围：成功率从 98.7% 掉到 91.4%，窗口 10:20-10:42。"""
    return {
        "metric": metric,
        "window": window,
        "anomaly_window": "10:20-10:42",
        "success_rate_before": "98.7%",
        "success_rate_now": "91.4%",
        "traffic": "稳定",
        "errors": "显著升高",
        "latency": "略升",
        "saturation": "未饱和",
    }


def query_traces(service="route-service"):
    """2. 失败环节：超时发生在打到已限流的兼容端点时，并且有重试放大。"""
    return {
        "service": service,
        "trace_id": "ab73...",
        "error": "PAYMENT_TIMEOUT",
        "path": "checkout → route-service → 兼容端点（已限流）",
        "note": "第一次超时后，重试把请求放大了",
    }


print("先手动看一眼指标，确认这些就是普通函数：")
show(query_metrics())


先手动看一眼指标，确认这些就是普通函数：
{
  "metric": "payment_success_rate",
  "window": "30m",
  "anomaly_window": "10:20-10:42",
  "success_rate_before": "98.7%",
  "success_rate_now": "91.4%",
  "traffic": "稳定",
  "errors": "显著升高",
  "latency": "略升",
  "saturation": "未饱和"
}


### 阶段二：定位根因，但同时寻找反证

看到超时还不能直接怪第三方。接下来从三个角度缩小范围：

1. 按版本和路由切片，找出异常组合；
2. 比较新旧配置，寻找能解释异常组合的直接证据；
3. 检查第三方状态，主动排除一个很容易想到的替代解释。


In [20]:
def query_success_by_slice(dimension="version_x_route"):
    """3. 触发条件：只有「新版本 + 新路由」成功率崩了。"""
    return {
        "dimension": dimension,
        "table": [
            {"version": "v2.3.6", "route": "旧路由", "success_rate": "98.8%"},
            {"version": "v2.3.6", "route": "新路由", "success_rate": "98.5%"},
            {"version": "v2.3.7", "route": "旧路由", "success_rate": "98.1%"},
            {"version": "v2.3.7", "route": "新路由", "success_rate": "72.6%"},
        ],
        "observation": "不像全网第三方故障，更像内部某条路径的问题",
    }


def compare_retry_config(old_version="v2.3.6", new_version="v2.3.7"):
    """4. 直接证据：10:17 的配置把重试目标指到了已限流端点。"""
    return {
        "change_time": "10:17",
        "old_version": old_version,
        "new_version": new_version,
        "old_retry_target": "主支付端点",
        "new_retry_target": "已限流的兼容端点",
        "scope": "仅 v2.3.7 在新路由读取该配置",
        "effect": "超时后重试放大，更多请求打到限流端点",
    }


def check_vendor_status(vendor="third_party_pay"):
    """5. 排除替代解释：第三方状态页正常，备用渠道也稳定。"""
    return {
        "vendor": vendor,
        "status_page": "正常",
        "backup_channel": "稳定",
        "global_degradation": False,
        "conclusion": "不是全局通道故障",
    }


### 阶段三：用修复结果验证因果

配置变更时间接近故障，只能说明“相关”；特定版本和路由命中错误配置，让解释更可信；如果回滚后指标恢复，证据链才进一步闭合。

这里的 `verify_rollback` 仍然只是**查询回滚结果**，不是让 Agent 自己执行回滚。


In [21]:
def verify_rollback(action="rollback_v2.3.7_route_config"):
    """6. 修复验证：回滚配置后，新路由成功率回到约 98%。"""
    return {
        "action": action,
        "after_rollback": "新路由成功率回到 98% 附近",
        "error_rate": "快速回落",
        "verified": True,
    }


## 3.3 不改循环，只替换工具和完成标准

下面先清空退款案例的工具，再登记六个支付查询工具，并把 6 问交给 Hook。

注意这里没有修改 `LoopPlugin.run()`：**业务知识进入工具和完成清单，通用执行逻辑留在 Harness。**


In [22]:
tools = kernel.get("tools")
tools.clear()

incident_list = [
    ("query_metrics", query_metrics, "查询支付成功率和异常时间窗口，确认影响范围。", {
        "metric": {"type": "string"},
        "window": {"type": "string"},
    }),
    ("query_traces", query_traces, "查询调用链和超时路径，定位失败环节。", {
        "service": {"type": "string"},
    }),
    ("query_success_by_slice", query_success_by_slice, "按版本和路由查看成功率，确认谁受影响。", {
        "dimension": {"type": "string"},
    }),
    ("compare_retry_config", compare_retry_config, "比较新旧版本的重试配置，寻找直接证据。", {
        "old_version": {"type": "string"},
        "new_version": {"type": "string"},
    }),
    ("check_vendor_status", check_vendor_status, "查询第三方支付通道是否故障。", {
        "vendor": {"type": "string"},
    }),
    ("verify_rollback", verify_rollback, "查询回滚后成功率是否恢复。", {
        "action": {"type": "string"},
    }),
]
for name, fn, desc, props in incident_list:
    tools.register(name, fn, make_schema(name, desc, props))

kernel.get("hook").required = {
    "query_metrics": {
        "label": "1. 影响范围",
        "result_fields": {"success_rate_before": "98.7%", "success_rate_now": "91.4%"},
    },
    "query_traces": {
        "label": "2. 失败环节",
        "result_fields": {"error": "PAYMENT_TIMEOUT"},
    },
    "query_success_by_slice": {
        "label": "3. 版本×路由触发条件",
        "result_fields": {"observation": "不像全网第三方故障，更像内部某条路径的问题"},
    },
    "compare_retry_config": {
        "label": "4. 直接配置证据",
        "result_fields": {"new_retry_target": "已限流的兼容端点"},
    },
    "check_vendor_status": {
        "label": "5. 排除第三方故障",
        "result_fields": {"global_degradation": False},
    },
    "verify_rollback": {
        "label": "6. 修复验证",
        "result_fields": {"verified": True},
    },
}

kernel.get("loop").system_prompt = """
你是线上支付故障排查助手。
规则：
1. 先确认影响，再定位失败环节；
2. 把想法当成假设，去找能支持或推翻它的证据；
3. 时间接近不等于因果；
4. 根据已有证据选择下一步工具，不要重复已经成功的调用；
5. 最终用中文写：现象、被排除的原因、根因、证据、建议；
6. 每个关键事实后标注 [证据: 工具名]，不得补充工具未返回的事实或时长承诺。
你可以在自认为查完时给出答案；能不能结束由系统检查。
""".strip()

print("完成清单（工具名、参数和关键返回字段都要有效）：")
show(kernel.get("hook").required)


  已登记工具：query_metrics
  已登记工具：query_traces
  已登记工具：query_success_by_slice
  已登记工具：compare_retry_config
  已登记工具：check_vendor_status
  已登记工具：verify_rollback
完成清单（工具名、参数和关键返回字段都要有效）：
{
  "query_metrics": {
    "label": "1. 影响范围",
    "result_fields": {
      "success_rate_before": "98.7%",
      "success_rate_now": "91.4%"
    }
  },
  "query_traces": {
    "label": "2. 失败环节",
    "result_fields": {
      "error": "PAYMENT_TIMEOUT"
    }
  },
  "query_success_by_slice": {
    "label": "3. 版本×路由触发条件",
    "result_fields": {
      "observation": "不像全网第三方故障，更像内部某条路径的问题"
    }
  },
  "compare_retry_config": {
    "label": "4. 直接配置证据",
    "result_fields": {
      "new_retry_target": "已限流的兼容端点"
    }
  },
  "check_vendor_status": {
    "label": "5. 排除第三方故障",
    "result_fields": {
      "global_degradation": false
    }
  },
  "verify_rollback": {
    "label": "6. 修复验证",
    "result_fields": {
      "verified": true
    }
  }
}


## 3.4 运行

不要把注意力放在工具调用顺序是否每次完全一样，而要沿着三阶段证据链观察：

1. **确认现象**：成功率何时下降，请求具体在哪一环超时？
2. **定位根因**：看到“只有新版本 + 新路由失败”后，模型会怎样更新假设？配置证据和第三方状态能否支持或推翻它？
3. **验证结论**：回滚后恢复，是否让“时间接近”升级为更完整的因果证据？

如果模型只说“和 v2.3.7 发布有关”就想结束，Hook 应该指出还缺的证据。


In [23]:
incident_run = kernel.get("loop").run(
    "线上支付成功率突然下降了，帮我查一下原因。请给出证据充分的结论。"
)

print("\n最终状态：", incident_run["status"])
print("\n最终答案：\n", incident_run["answer"])



========== 第 1 轮 ==========
模型申请调用：query_metrics
参数： {'metric': 'payment_success_rate', 'window': '30m'}
Harness 执行结果：
{
  "metric": "payment_success_rate",
  "window": "30m",
  "anomaly_window": "10:20-10:42",
  "success_rate_before": "98.7%",
  "success_rate_now": "91.4%",
  "traffic": "稳定",
  "errors": "显著升高",
  "latency": "略升",
  "saturation": "未饱和"
}

========== 第 2 轮 ==========
模型申请调用：query_traces
参数： {'service': 'route-service'}
Harness 执行结果：
{
  "service": "route-service",
  "trace_id": "ab73...",
  "error": "PAYMENT_TIMEOUT",
  "path": "checkout → route-service → 兼容端点（已限流）",
  "note": "第一次超时后，重试把请求放大了"
}

========== 第 3 轮 ==========
模型申请调用：query_success_by_slice
参数： {'dimension': 'version_x_route'}
Harness 执行结果：
{
  "dimension": "version_x_route",
  "table": [
    {
      "version": "v2.3.6",
      "route": "旧路由",
      "success_rate": "98.8%"
    },
    {
      "version": "v2.3.6",
      "route": "新路由",
      "success_rate": "98.5%"
    },
    {
      "version": "v2.3.7",
  

## 3.5 回放：答案是“说了什么”，流水账是“怎么做到的”

最终答案面向用户；回放记录面向开发者和系统审计。两者缺一不可。


In [24]:
print("流水账：")
kernel.get("session").replay()

print("\n记录条数统计：")
show(dict(Counter(event["type"] for event in kernel.get("session").events)))


流水账：
  新运行已创建
  第 1 轮  申请调用 query_metrics  {'metric': 'payment_success_rate', 'window': '30m'}
  第 1 轮  工具返回 query_metrics  成功=True
  第 2 轮  申请调用 query_traces  {'service': 'route-service'}
  第 2 轮  工具返回 query_traces  成功=True
  第 3 轮  申请调用 query_success_by_slice  {'dimension': 'version_x_route'}
  第 3 轮  工具返回 query_success_by_slice  成功=True
  第 4 轮  申请调用 compare_retry_config  {'old_version': 'v2.3.6', 'new_version': 'v2.3.7'}
  第 4 轮  工具返回 compare_retry_config  成功=True
  第 5 轮  申请调用 check_vendor_status  {'vendor': 'third_party_pay'}
  第 5 轮  工具返回 check_vendor_status  成功=True
  第 6 轮  申请调用 verify_rollback  {'action': 'rollback_v2.3.7_route_config'}
  第 6 轮  工具返回 verify_rollback  成功=True
  第 7 轮  完成检查通过
  第 7 轮  任务结束，原因：completed

记录条数统计：
{
  "user_message": 1,
  "tool/call": 6,
  "tool/result": 6,
  "hook/pass": 1,
  "turn/end": 1
}


## 3.6 Evaluation：通过门禁，不等于答案一定高质量

当运行状态为 `completed` 时，Hook 已保证六类有效证据都被查询过；Evaluation 再从两个角度验收：

- **过程分**：六项有效证据是不是都属于当前 run_id；
- **结果分**：最终文字是否引用关键证据、组织出根因和建议，并避免加入证据之外的时长承诺。

教学版仍然是轻量规则检查。真实项目通常还会加入事实一致性、证据引用解析、失败用例、人工标注样本和多次运行稳定性评估。


In [25]:
done = kernel.get("session").successful_tools()
answer = incident_run.get("answer") or ""
unsupported_phrases = ["1-2 个工作日", "几分钟到几小时", "银行清算周期"]

evaluation = kernel.get("eval").evaluate(
    process_checks={
        "过程：查了影响范围": "query_metrics" in done,
        "过程：查了失败环节": "query_traces" in done,
        "过程：查了版本×路由": "query_success_by_slice" in done,
        "过程：查了配置差异": "compare_retry_config" in done,
        "过程：排除了第三方": "check_vendor_status" in done,
        "过程：验证了回滚": "verify_rollback" in done,
    },
    result_checks={
        "答案：引用指标证据": "[证据: query_metrics]" in answer,
        "答案：引用配置证据": "[证据: compare_retry_config]" in answer,
        "答案：引用回滚证据": "[证据: verify_rollback]" in answer,
        "答案：没有无证据时长承诺": not any(text in answer for text in unsupported_phrases),
    },
)
show(evaluation)


{
  "score": 100,
  "checks": {
    "过程：查了影响范围": true,
    "过程：查了失败环节": true,
    "过程：查了版本×路由": true,
    "过程：查了配置差异": true,
    "过程：排除了第三方": true,
    "过程：验证了回滚": true,
    "答案：引用指标证据": true,
    "答案：引用配置证据": true,
    "答案：引用回滚证据": true,
    "答案：没有无证据时长承诺": true
  }
}


## 3.7 以后换场景，保留的是这五层，不是这六个函数

| 层 | 这次用到了什么 | 换到别的故障时仍然要有 |
|---|---|---|
| 只读工具 | 指标、链路、切片、配置、厂商、回滚查询 | 默认不要有副作用 |
| Session | `events` 流水账 | 能回放 |
| Loop | 假设 → 取证 → 更新 | 下一步由结果决定 |
| Hook | 6 问缺一不可 | 证据不够就不能停 |
| Evaluation | 过程 + 结果 | 能验收 |

工具会随业务变化，真正能搬到下一场景的是：**状态、循环、门禁、评估**。

换场景时，可以按这个顺序设计：

1. 先写“怎样才算完成”；
2. 再准备取得这些证据所需的工具；
3. 最后让同一个 Loop 去运行。


# 练习

每次只改一样东西，这样才知道是哪一层在起作用。

**练习 1**  
在 `query_traces` 的返回值里加一条只出现 5 次的数据库错误。重新运行，观察模型会不会被小样本带偏。

**练习 2**  
新增 `check_database_status`（固定返回正常），登记到 Tools，并加入 Hook 清单。观察“新增一种必需证据”要改哪些地方。

**练习 3**  
从 Hook 清单里删掉 `verify_rollback`，但保留 Evaluation 对回滚证据的检查。观察 Agent 是否更早结束，以及评分器是否仍会扣分。用这个实验解释 Hook 和 Evaluation 的区别。

**练习 4**  
如果以后增加“真正回滚配置”的写操作，审批应该加在 Tools 的“能不能做”，不要指望 Prompt 中一句“请谨慎”。


# 小结

开头的六级操作台阶，最终收束成三个核心认识：

1. 从普通聊天走到 Tool Calls，再走到**根据工具结果决定下一步**的 Agent Loop；
2. 看到裸 Loop 会把停止权完全交给模型，因此用 Harness 增加参数校验、运行隔离、可信记账、证据门禁和验收；
3. 在支付排障中只替换工具和完成标准，复用同一套教学版 Harness 主链。

请再复述三句话：

> **Agent 不是一次模型调用，而是一个由结果推动下一步的循环。**

> **Tool Calls 是调用申请，真正执行工具的是模型外面的程序。**

> **Harness 决定模型看到什么、能做什么、何时继续或停止，以及如何记录和验收。**

以后遇到 Agent 项目，可以先问：

1. 怎样才算做完？完成标准是否包含任务、参数和有效结果？
2. 调过什么、返回了什么、属于哪次运行、为什么能停，能回放吗？
3. 最终答案的关键事实能否追溯到具体证据？

**不要只问“模型能不能做”，还要问“系统能不能让它可靠地做完”。**

## 与官方 DeepSeek Harness 的关系

本 Notebook 用极小的 Python 类讲解核心思想，没有直接使用官方 DeepSeek Harness。官方项目仍处于开发者预览阶段，包含模型、工具、技能、会话、沙箱、存储、循环、调度和 UI 等更多组件，接口可能继续演进。

## 资料

- [DeepSeek Tool Calls](https://api-docs.deepseek.com/guides/tool_calls/)
- [DeepSeek Thinking Mode](https://api-docs.deepseek.com/guides/thinking_mode/)
- [DeepSeek Harness 官方介绍](https://github.com/deepseek-ai/deepseek-harness)
- [DeepSeek Harness 开发者文档](https://deepseek-harness.github.io/)
